In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression

# Load dataset
df = pd.read_csv("../data/archive/creditcard.csv")

# Remove duplicates
df = df.drop_duplicates().copy()

# Separate features and target
X = df.drop("Class", axis=1)
y = df["Class"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale Time and Amount
scaler = StandardScaler()

X_train[["Time", "Amount"]] = scaler.fit_transform(
    X_train[["Time", "Amount"]]
)

X_test[["Time", "Amount"]] = scaler.transform(
    X_test[["Time", "Amount"]]
)

# Apply SMOTE only to training data
smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train, y_train
)

print("Data preparation completed!")
print("Training data:", X_train_resampled.shape)
print("Testing data:", X_test.shape)

Data preparation completed!
Training data: (453204, 30)
Testing data: (56746, 30)


In [4]:
# Initialize Logistic Regression
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model
logistic_model.fit(
    X_train_resampled,
    y_train_resampled
)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [5]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    average_precision_score
)

# Predict fraud or genuine transactions
y_pred = logistic_model.predict(X_test)

# Get fraud prediction probabilities
y_pred_proba = logistic_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("PR-AUC:", average_precision_score(y_test, y_pred_proba))

Accuracy: 0.9736721531033025

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56651
           1       0.05      0.87      0.10        95

    accuracy                           0.97     56746
   macro avg       0.53      0.92      0.54     56746
weighted avg       1.00      0.97      0.99     56746


Confusion Matrix:
[[55169  1482]
 [   12    83]]

ROC-AUC: 0.9618957810936585
PR-AUC: 0.6768816018011278


## Logistic Regression - Results

- Accuracy: 97.37%
- Fraud Precision: 5%
- Fraud Recall: 87%
- Fraud F1-score: 0.10
- ROC-AUC: 0.9619
- PR-AUC: 0.6769

### Observations
1. The model detected 83 out of 95 fraudulent transactions.
2. It missed 12 fraudulent transactions.
3. It incorrectly flagged 1,482 genuine transactions.
4. Fraud recall is good, but fraud precision is very low.
5. The model generates too many false positives.

Conclusion: Logistic Regression provides a useful baseline, but its fraud predictions need improvement.

In [6]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Train the model
random_forest_model.fit(
    X_train_resampled,
    y_train_resampled
)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [7]:
# Predict fraud or genuine transactions
rf_y_pred = random_forest_model.predict(X_test)

# Get prediction probabilities
rf_y_pred_proba = random_forest_model.predict_proba(X_test)[:, 1]

# Evaluate the model
print("Random Forest Results")
print("---------------------")

print("Accuracy:", accuracy_score(y_test, rf_y_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, rf_y_pred_proba))
print("PR-AUC:", average_precision_score(y_test, rf_y_pred_proba))

Random Forest Results
---------------------
Accuracy: 0.9991012582384662

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.72      0.77      0.74        95

    accuracy                           1.00     56746
   macro avg       0.86      0.88      0.87     56746
weighted avg       1.00      1.00      1.00     56746


Confusion Matrix:
[[56622    29]
 [   22    73]]

ROC-AUC: 0.978929902291872
PR-AUC: 0.7954005483458295


## Model Comparison

Two machine learning algorithms were trained and evaluated:

1. Logistic Regression
2. Random Forest Classifier

### Observations

- Random Forest achieved 99.91% accuracy.
- Random Forest achieved 72% fraud precision.
- Random Forest detected 73 out of 95 fraudulent transactions.
- Logistic Regression detected 83 out of 95 fraudulent transactions.
- Random Forest reduced false positives from 1,482 to 29.
- Random Forest achieved a better F1-score and PR-AUC.

### Conclusion

Random Forest performed better overall, providing a stronger balance between fraud precision and recall.

However, Logistic Regression achieved higher fraud recall. Further experimentation with model parameters and classification thresholds may improve the results.